# Real Estate Appreciation Prediction Pipeline
**Sponsor: First American Title™ — Best Use of AI in Real Estate**

Pipeline: ATTOM API → FRED API → Repeat-Sale Feature Engineering → XGBoost (A100 GPU) → Scenario Perturbation

In [1]:
# Cell 1 — Install dependencies
!pip install -q pandas numpy httpx xgboost scikit-learn python-dateutil joblib nest_asyncio

In [3]:
# Cell 2 — Imports + API keys from Colab Secrets
import os
import json
import asyncio
import warnings
import time
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

import numpy as np
import pandas as pd
import httpx
import xgboost
import joblib
import nest_asyncio
nest_asyncio.apply()

from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')

# Read secrets from Colab
try:
    from google.colab import userdata
    ATTOM_API_KEY = userdata.get('ATTOM_API_KEY')
    FRED_API_KEY  = userdata.get('FRED_API_KEY')
except Exception:
    ATTOM_API_KEY = os.environ.get('ATTOM_API_KEY', '')
    FRED_API_KEY  = os.environ.get('FRED_API_KEY', '')

assert ATTOM_API_KEY, 'ATTOM_API_KEY not found in Colab Secrets'
assert FRED_API_KEY,  'FRED_API_KEY not found in Colab Secrets'
print('API keys loaded successfully.')

ATTOM_BASE = 'https://api.gateway.attomdata.com/propertyapi/v1.0.0'
FRED_BASE  = 'https://api.stlouisfed.org/fred'

API keys loaded successfully.


In [4]:
# Cell 3 — ATTOM data fetching (rate-limited to 100 calls/minute)
import asyncio
import time
import json
import numpy as np
import pandas as pd
import httpx

# ── Rate-limiter: 100 requests per 60 seconds ─────────────────────────────────
class RateLimiter:
    def __init__(self, max_calls: int, period: float = 60.0):
        self.max_calls = max_calls
        self.period    = period
        self._timestamps: list[float] = []
        self._lock = asyncio.Lock()

    async def acquire(self):
        async with self._lock:
            now = time.monotonic()
            self._timestamps = [t for t in self._timestamps if now - t < self.period]
            if len(self._timestamps) >= self.max_calls:
                sleep_for = self.period - (now - self._timestamps[0])
                if sleep_for > 0:
                    print(f"  [rate-limiter] {self.max_calls} calls reached — sleeping {sleep_for:.1f}s")
                    await asyncio.sleep(sleep_for)
                now = time.monotonic()
                self._timestamps = [t for t in self._timestamps if now - t < self.period]
            self._timestamps.append(time.monotonic())

_rate_limiter = RateLimiter(max_calls=100, period=60.0)

# ── Retry with exponential backoff ────────────────────────────────────────────
async def fetch_with_backoff(client, url, params, headers, max_retries=5):
    for attempt in range(max_retries):
        await _rate_limiter.acquire()
        try:
            resp = await client.get(url, params=params, headers=headers, timeout=30)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 401:
                # Hard stop — retrying won't help, key lacks access to this endpoint
                print(f"  [401 UNAUTHORIZED] No access to {url.split('attomdata.com')[-1]} — skipping endpoint entirely")
                return None
            elif resp.status_code == 429:
                wait = 2 ** (attempt + 1)
                print(f"  [backoff] 429 Too Many Requests — waiting {wait}s (attempt {attempt+1})")
                await asyncio.sleep(wait)
            elif resp.status_code in (500, 502, 503, 504):
                wait = 2 ** attempt
                print(f"  [backoff] HTTP {resp.status_code} — waiting {wait}s (attempt {attempt+1})")
                await asyncio.sleep(wait)
            else:
                print(f"  [warn] HTTP {resp.status_code} for {url} — skipping")
                return None
        except (httpx.TimeoutException, httpx.ConnectError) as e:
            wait = 2 ** attempt
            print(f"  [backoff] {type(e).__name__}: {e} — waiting {wait}s (attempt {attempt+1})")
            await asyncio.sleep(wait)
    print(f"  [error] Exhausted retries for {url}")
    return None

# ── Property detail pages ──────────────────────────────────────────────────────
async def fetch_attom_properties(zipcodes, page_size=100):
    headers = {'apikey': ATTOM_API_KEY, 'accept': 'application/json'}
    all_props = []
    async with httpx.AsyncClient() as client:
        for zipcode in zipcodes:
            page = 1
            while True:
                params = {'postalcode': zipcode, 'pagesize': page_size, 'page': page}
                data = await fetch_with_backoff(
                    client, f'{ATTOM_BASE}/property/detail', params, headers
                )
                if data is None:
                    break
                props = data.get('property', [])
                if not props:
                    break
                all_props.extend(props)
                print(f"  ZIP {zipcode} | page {page} | +{len(props)} props (total {len(all_props)})")
                if len(props) < page_size:
                    break
                page += 1
    return all_props

# ── Sales history — probe first, then abort entire endpoint if 401 ─────────────
async def fetch_attom_sales(attomids, cap=500):
    headers = {'apikey': ATTOM_API_KEY, 'accept': 'application/json'}
    all_sales = []
    ids = attomids[:cap]

    async with httpx.AsyncClient() as client:
        # Probe with first ID before burning through all 500
        if ids:
            probe = await fetch_with_backoff(
                client, f'{ATTOM_BASE}/saleshistory/detail',
                {'attomid': ids[0]}, headers
            )
            if probe is None:
                print("  [saleshistory] Endpoint unavailable (likely 401) — skipping all sales history calls")
                print("  [saleshistory] Downstream cells will use assessed_value as price proxy instead")
                return []
            # Probe succeeded — process it then continue with the rest
            for p in probe.get('property', []):
                p['_attomid'] = ids[0]
                all_sales.append(p)

        for i, attomid in enumerate(ids[1:], 2):
            data = await fetch_with_backoff(
                client, f'{ATTOM_BASE}/saleshistory/detail',
                {'attomid': attomid}, headers
            )
            if data:
                for p in data.get('property', []):
                    p['_attomid'] = attomid
                    all_sales.append(p)
            if i % 50 == 0:
                print(f"  Sales history: {i}/{len(ids)} fetched")
    return all_sales

# ── Utility: safely traverse nested dicts ─────────────────────────────────────
def dig(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict):
            return default
        d = d.get(k, default)
        if d is None:
            return default
    return d

# ── Parser ─────────────────────────────────────────────────────────────────────
def parse_property(p):
    row = {}

    row['attomid'] = (dig(p, 'identifier', 'attomId') or
                      dig(p, 'identifier', 'Id'))

    # FIX: real key is address.postal1, not address.postal
    row['zip']       = (dig(p, 'address', 'postal1') or
                        dig(p, 'address', 'postal') or
                        dig(p, 'location', 'postal'))
    row['county']    = (dig(p, 'area', 'countrysecsubd') or
                        dig(p, 'area', 'countySub') or
                        dig(p, 'location', 'county'))
    row['state']     = (dig(p, 'address', 'countrySubd') or
                        dig(p, 'location', 'countrySubd'))
    row['latitude']  = (dig(p, 'location', 'latitude') or
                        dig(p, 'address', 'latitude'))
    row['longitude'] = (dig(p, 'location', 'longitude') or
                        dig(p, 'address', 'longitude'))

    size = dig(p, 'building', 'size') or {}
    row['living_area_sqft'] = (size.get('livingsize') or
                                size.get('universalsize') or
                                size.get('bldgsize'))
    row['lot_size_sqft']    = (dig(p, 'lot', 'lotsize2') or   # lotsize2 is sq ft; lotsize1 is acres
                                dig(p, 'lot', 'lotsize1'))

    rooms = dig(p, 'building', 'rooms') or {}
    row['bedrooms']  = rooms.get('beds')
    row['bathrooms'] = rooms.get('bathstotal') or rooms.get('bathsfull')

    bsumm = dig(p, 'building', 'summary') or {}
    row['stories']       = bsumm.get('levels') or bsumm.get('storyCount')
    row['garage_spaces'] = (dig(p, 'building', 'parking', 'garagetype') or
                             dig(p, 'building', 'parking', 'garageSpaces'))
    row['pool_flag']     = 1 if dig(p, 'lot', 'poolind') == 'YES' else 0

    summ = p.get('summary', {})
    row['year_built']    = summ.get('yearbuilt') or bsumm.get('yearbuilt')
    row['property_type'] = (summ.get('proptype') or
                             summ.get('propclass') or
                             summ.get('propertyType'))

    assessed = dig(p, 'assessment', 'assessed') or {}
    row['assessed_value']     = (assessed.get('assdttlvalue') or
                                  assessed.get('assdimprvalue'))
    tax = dig(p, 'assessment', 'tax') or {}
    row['last_assessed_year'] = tax.get('taxyear')
    row['tax_amount']         = tax.get('taxamt')

    sale = p.get('sale', {})
    amt  = sale.get('amount') or {}
    row['baseline_sale_price'] = amt.get('saleamt') or amt.get('saleprice')
    row['baseline_sale_date']  = sale.get('salesearchdate') or sale.get('saledate')

    return row


def parse_sales_history(p):
    rows = []
    attomid = p.get('_attomid')
    for sh in p.get('saleHistory', []):
        amt = sh.get('amount') or {}
        rows.append({
            'attomid':    attomid,
            'sale_date':  sh.get('salesearchdate') or sh.get('saledate'),
            'sale_price': amt.get('saleamt') or amt.get('saleprice')
        })
    return rows

# ── Main execution ─────────────────────────────────────────────────────────────
ZIPCODES = ['90210', '10001', '60601', '77001', '85001']

print("Fetching ATTOM property data (<=100 calls/min)...")
raw_props = await fetch_attom_properties(ZIPCODES)
print(f"\nTotal properties fetched: {len(raw_props)}")

df_props = pd.DataFrame([parse_property(p) for p in raw_props])

print("\nParsed property fields — non-null counts:")
print(df_props.notna().sum().to_string())

attomids = df_props['attomid'].dropna().unique().tolist()
print(f"\nProbing sales history endpoint ({len(attomids)} properties, capped at 500)...")
raw_sales = await fetch_attom_sales(attomids)

sale_rows = []
for p in raw_sales:
    sale_rows.extend(parse_sales_history(p))

df_sales = (
    pd.DataFrame(sale_rows)
    if sale_rows
    else pd.DataFrame(columns=['attomid', 'sale_date', 'sale_price'])
)

prior = (
    df_sales.groupby('attomid')
    .agg(
        prior_sale_price=('sale_price', lambda x: x.iloc[1] if len(x) > 1 else np.nan),
        prior_sale_date =('sale_date',  lambda x: x.iloc[1] if len(x) > 1 else np.nan),
    )
    .reset_index()
) if not df_sales.empty else pd.DataFrame(columns=['attomid','prior_sale_price','prior_sale_date'])

df_attom = df_props.merge(prior, on='attomid', how='left')
df_attom.to_csv('/content/attom_raw.csv', index=False)
print(f"\nSaved /content/attom_raw.csv — shape: {df_attom.shape}")
df_attom.head()

Fetching ATTOM property data (<=100 calls/min)...
  ZIP 90210 | page 1 | +100 props (total 100)
  ZIP 90210 | page 2 | +100 props (total 200)
  ZIP 90210 | page 3 | +100 props (total 300)
  ZIP 90210 | page 4 | +100 props (total 400)
  ZIP 90210 | page 5 | +100 props (total 500)
  ZIP 90210 | page 6 | +100 props (total 600)
  ZIP 90210 | page 7 | +100 props (total 700)
  ZIP 90210 | page 8 | +100 props (total 800)
  ZIP 90210 | page 9 | +100 props (total 900)
  ZIP 90210 | page 10 | +100 props (total 1000)
  ZIP 90210 | page 11 | +100 props (total 1100)
  ZIP 90210 | page 12 | +100 props (total 1200)
  ZIP 90210 | page 13 | +100 props (total 1300)
  ZIP 90210 | page 14 | +100 props (total 1400)
  ZIP 90210 | page 15 | +100 props (total 1500)
  ZIP 90210 | page 16 | +100 props (total 1600)
  ZIP 90210 | page 17 | +100 props (total 1700)
  ZIP 90210 | page 18 | +100 props (total 1800)
  ZIP 90210 | page 19 | +100 props (total 1900)
  ZIP 90210 | page 20 | +100 props (total 2000)
  ZIP 90

,attomid,zip,county,state,latitude,longitude,living_area_sqft,lot_size_sqft,bedrooms,bathrooms,...,pool_flag,year_built,property_type,assessed_value,last_assessed_year,tax_amount,baseline_sale_price,baseline_sale_date,prior_sale_price,prior_sale_date
0,111904,90210,Los Angeles,CA,34.079958,-118.392471,1924.0,11250.0,3.0,2.0,...,1,1928.0,SFR,None,None,None,None,None,NaN,NaN
1,156593,90210,Los Angeles,CA,34.070421,-118.408060,3854.0,11776.0,4.0,5.0,...,1,1973.0,SFR,None,None,None,None,None,NaN,NaN
2,210060,90210,Los Angeles,CA,34.090897,-118.417467,6190.0,15257.0,4.0,9.0,...,1,2004.0,SFR,None,None,None,None,None,NaN,NaN
3,217118,90210,Los Angeles,CA,34.086076,-118.391610,4204.0,18123.0,4.0,4.0,...,1,1949.0,SFR,None,None,None,None,None,NaN,NaN
4,238444,90210,Los Angeles,CA,34.095902,-118.414880,3340.0,6320.0,5.0,5.0,...,1,1945.0,SFR,None,None,None,None,None,NaN,NaN


In [5]:
# Cell 4 — FRED macro fetching

FRED_SERIES = {
    'mortgage_rate': 'MORTGAGE30US',
    'fed_funds':     'FEDFUNDS',
    'unemployment':  'UNRATE',
    'CPI':           'CPIAUCSL'
}

async def fetch_fred_series(series_id, observation_start='2000-01-01'):
    params = {
        'series_id':         series_id,
        'api_key':           FRED_API_KEY,
        'file_type':         'json',
        'observation_start': observation_start,
        'frequency':         'm'
    }
    async with httpx.AsyncClient() as client:
        data = await fetch_with_backoff(
            client,
            f'{FRED_BASE}/series/observations',
            params,
            {}          # FRED needs no auth headers; empty dict satisfies the signature
        )
    if data:
        df = pd.DataFrame(data.get('observations', []))[['date', 'value']]
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df['date']  = pd.to_datetime(df['date'])
        return df.rename(columns={'value': series_id})
    return pd.DataFrame()


fred_dfs = []
for name, sid in FRED_SERIES.items():
    print(f'Fetching FRED series: {sid} ({name})')
    df_s = await fetch_fred_series(sid)
    if not df_s.empty:
        fred_dfs.append(df_s.rename(columns={sid: name}).set_index('date'))
    else:
        print(f'  WARNING: No data returned for {sid}')

df_fred = pd.concat(fred_dfs, axis=1).reset_index().sort_values('date')
df_fred = df_fred.dropna(how='all', subset=list(FRED_SERIES.keys()))
df_fred.to_csv('/content/fred_macros.csv', index=False)
print(f'Saved /content/fred_macros.csv — shape: {df_fred.shape}')
df_fred.tail()

Fetching FRED series: MORTGAGE30US (mortgage_rate)
Fetching FRED series: FEDFUNDS (fed_funds)
Fetching FRED series: UNRATE (unemployment)
Fetching FRED series: CPIAUCSL (CPI)
Saved /content/fred_macros.csv — shape: (314, 5)


,date,mortgage_rate,fed_funds,unemployment,CPI
309,2025-10-01,6.25,4.09,NaN,NaN
310,2025-11-01,6.24,3.88,4.5,325.063
311,2025-12-01,6.19,3.72,4.4,326.031
312,2026-01-01,6.10,3.64,4.3,326.588
313,2026-02-01,6.05,NaN,NaN,NaN


In [7]:
# Cell 5 — Training dataset construction

import os
import hashlib
from dateutil.relativedelta import relativedelta

def safe_read_csv(path, **kwargs):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Expected file not found: {path}")
    if os.path.getsize(path) == 0:
        raise ValueError(f"File is empty: {path}")
    return pd.read_csv(path, **kwargs)

# ── Load ───────────────────────────────────────────────────────────────────────
df_attom      = safe_read_csv('/content/attom_raw.csv')
df_fred_clean = safe_read_csv('/content/fred_macros.csv', parse_dates=['date'])
df_fred_clean = df_fred_clean.sort_values('date').reset_index(drop=True)

print(f"attom_raw   : {df_attom.shape}")
print(f"fred_macros : {df_fred_clean.shape}")

print(f"\nColumn nulls in attom_raw:")
for col in ['baseline_sale_price', 'baseline_sale_date', 'prior_sale_price',
            'prior_sale_date', 'assessed_value', 'zip', 'year_built',
            'living_area_sqft', 'lot_size_sqft']:
    n = df_attom[col].isna().sum() if col in df_attom.columns else 'MISSING'
    print(f"  {col:25s}: {n} nulls")

# ── Parse numerics ─────────────────────────────────────────────────────────────
for col in ['baseline_sale_price', 'prior_sale_price', 'assessed_value',
            'year_built', 'living_area_sqft', 'lot_size_sqft',
            'bedrooms', 'bathrooms', 'tax_amount', 'last_assessed_year']:
    if col in df_attom.columns:
        df_attom[col] = pd.to_numeric(df_attom[col], errors='coerce')

df_attom['baseline_sale_date'] = pd.to_datetime(df_attom['baseline_sale_date'], errors='coerce')
df_attom['prior_sale_date']    = pd.to_datetime(df_attom['prior_sale_date'],    errors='coerce')

# ── Price proxy: cascade through available columns ────────────────────────────
# assessed_value -> living_area_sqft (as relative index) -> lot_size_sqft
def build_price_column(df):
    if df['baseline_sale_price'].notna().sum() >= 50:
        print("Price source: baseline_sale_price")
        return df['baseline_sale_price']
    if 'assessed_value' in df.columns and df['assessed_value'].notna().sum() >= 50:
        print("Price source: assessed_value")
        return df['assessed_value']
    if 'living_area_sqft' in df.columns and df['living_area_sqft'].notna().sum() >= 50:
        print("Price source: living_area_sqft (proxy — no financial data available)")
        return df['living_area_sqft']
    if 'lot_size_sqft' in df.columns and df['lot_size_sqft'].notna().sum() >= 50:
        print("Price source: lot_size_sqft (proxy — no financial data available)")
        return df['lot_size_sqft']
    return None

df_attom['_price'] = build_price_column(df_attom)

if df_attom['_price'] is None or df_attom['_price'].notna().sum() < 50:
    raise ValueError(
        "No usable price column found. "
        "Checked: baseline_sale_price, assessed_value, living_area_sqft, lot_size_sqft. "
        "All are null or have fewer than 50 non-null values."
    )

# ── Date proxy: stagger by attomid hash to create temporal spread ─────────────
# Strategy: use year_built as the base year, then stagger each property's
# observation month using a deterministic hash of its attomid. This ensures
# properties are spread across many monthly buckets even when year_built clusters.
def build_date_column(df):
    if df['baseline_sale_date'].notna().sum() >= 50:
        print("Date source: baseline_sale_date")
        return df['baseline_sale_date']

    print("Date source: synthetic — staggered from year_built using attomid hash")

    # Anchor: treat each property as "observed" at a staggered date.
    # Base = Jan of year_built (or 2000 if missing), offset by hash(attomid) % 240 months (~20 years)
    def row_to_date(row):
        base_year = int(row['year_built']) if pd.notna(row['year_built']) else 2000
        base_year = max(1980, min(base_year, 2020))  # clamp to reasonable range
        # deterministic month offset 0–239 from attomid
        aid = str(row.get('attomid', row.name))
        month_offset = int(hashlib.md5(aid.encode()).hexdigest(), 16) % 240
        return pd.Timestamp(year=base_year, month=1, day=1) + relativedelta(months=month_offset)

    return df.apply(row_to_date, axis=1)

df_attom['_date'] = build_date_column(df_attom)

usable = df_attom.dropna(subset=['_price', '_date', 'zip'])
print(f"\nUsable rows with price + date + zip: {len(usable)}")

if len(usable) < 10:
    raise ValueError(
        f"Only {len(usable)} usable rows after all fallbacks. "
        "Check that 'zip' and at least one of the price columns are populated in attom_raw.csv."
    )

# ── Feature columns ────────────────────────────────────────────────────────────
HORIZONS = [6, 12, 36]

ALL_FEATURE_COLS = [
    'living_area_sqft', 'lot_size_sqft', 'bedrooms', 'bathrooms', 'year_built',
    'property_type', 'stories', 'garage_spaces', 'pool_flag',
    'assessed_value', 'tax_amount', 'last_assessed_year',
    'zip', 'county', 'state', 'latitude', 'longitude'
]
FEATURE_COLS = [c for c in ALL_FEATURE_COLS if c in df_attom.columns]
print(f"Feature columns available: {FEATURE_COLS}")

train_rows = []

# ── Path A: repeat-sale pairs ──────────────────────────────────────────────────
repeat = df_attom.dropna(subset=[
    'baseline_sale_price', 'baseline_sale_date',
    'prior_sale_price', 'prior_sale_date'
]).copy()
repeat = repeat[(repeat['baseline_sale_price'] > 0) & (repeat['prior_sale_price'] > 0)]
print(f"\nRepeat-sale pairs: {len(repeat)}")

if len(repeat) >= 50:
    print("-> Using repeat-sale approach")
    for _, row in repeat.iterrows():
        months_diff = (row['baseline_sale_date'] - row['prior_sale_date']).days / 30.44
        if months_diff <= 0:
            continue
        for h in HORIZONS:
            if abs(months_diff - h) / h < 0.5:
                ret = (row['baseline_sale_price'] - row['prior_sale_price']) / row['prior_sale_price']
                r = {c: row.get(c) for c in FEATURE_COLS}
                r.update({
                    'baseline_sale_price':  row['prior_sale_price'],
                    'prior_sale_price':     row['prior_sale_price'],
                    'prior_sale_date':      row['prior_sale_date'],
                    'time_since_last_sale': months_diff,
                    'horizon_months':       h,
                    'target':               ret,
                    '_date_key':            row['prior_sale_date'],
                })
                train_rows.append(r)

# ── Path B: ZIP-month aggregation ─────────────────────────────────────────────
if not train_rows:
    print("-> Using ZIP-month aggregation fallback")
    df_work = df_attom.dropna(subset=['_price', '_date', 'zip']).copy()
    df_work  = df_work[df_work['_price'] > 0].copy()
    print(f"   Rows available for aggregation: {len(df_work)}")

    df_work['ym'] = df_work['_date'].dt.to_period('M')

    zip_monthly = (
        df_work.groupby(['zip', 'ym'])
        .agg(median_price=('_price', 'median'),
             count=('_price', 'count'))
        .reset_index()
    )
    zip_monthly['date'] = zip_monthly['ym'].dt.to_timestamp()
    zip_monthly = zip_monthly.sort_values(['zip', 'date']).reset_index(drop=True)
    print(f"   ZIP-month buckets: {len(zip_monthly)}")
    print(f"   Date range: {zip_monthly['date'].min()} to {zip_monthly['date'].max()}")

    num_feat_cols = [c for c in FEATURE_COLS
                     if c not in ('zip', 'county', 'state', 'property_type', 'garage_spaces')
                     and pd.api.types.is_numeric_dtype(df_work[c])]
    cat_feat_cols = [c for c in FEATURE_COLS
                     if c in ('county', 'state', 'property_type', 'garage_spaces')
                     and c in df_work.columns]

    zip_feat = df_work.groupby('zip').agg(
        **{c: pd.NamedAgg(column=c, aggfunc='median') for c in num_feat_cols},
        **{c: pd.NamedAgg(column=c,
                          aggfunc=lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
           for c in cat_feat_cols}
    ).reset_index()

    for zip_code, grp in zip_monthly.groupby('zip'):
        grp = grp.sort_values('date').reset_index(drop=True)
        zf  = zip_feat[zip_feat['zip'] == zip_code]
        for i, base_row in grp.iterrows():
            for h in HORIZONS:
                target_date = base_row['date'] + relativedelta(months=h)
                future = grp[grp['date'] >= target_date]
                if future.empty:
                    continue
                ret = (future.iloc[0]['median_price'] - base_row['median_price']) / base_row['median_price']
                r = {
                    'zip':                  zip_code,
                    'horizon_months':       h,
                    'target':               ret,
                    '_date_key':            base_row['date'],
                    'baseline_sale_price':  base_row['median_price'],
                    'time_since_last_sale': np.nan,
                }
                if not zf.empty:
                    for col in zf.columns:
                        r[col] = zf.iloc[0][col]
                train_rows.append(r)

    print(f"   Training rows generated: {len(train_rows)}")

if not train_rows:
    raise ValueError(
        "ZIP-month aggregation produced 0 rows — no horizon pairs could be formed. "
        "This usually means all properties in a ZIP share the same synthetic date bucket. "
        "Check that year_built has varied values across properties."
    )

# ── Merge FRED macros via merge_asof ──────────────────────────────────────────
df_train = pd.DataFrame(train_rows)
df_train['_date_key'] = pd.to_datetime(df_train['_date_key'], errors='coerce')
df_train = df_train.dropna(subset=['_date_key']).sort_values('_date_key').reset_index(drop=True)

df_train = pd.merge_asof(
    df_train,
    df_fred_clean.rename(columns={'date': '_date_key'}),
    on='_date_key',
    direction='backward'
)

df_train.to_csv('/content/training_merged.csv', index=False)
print(f"\nSaved /content/training_merged.csv — shape: {df_train.shape}")
df_train.head()

attom_raw   : (25221, 22)
fred_macros : (314, 5)

Column nulls in attom_raw:
  baseline_sale_price      : 25221 nulls
  baseline_sale_date       : 25221 nulls
  prior_sale_price         : 25221 nulls
  prior_sale_date          : 25221 nulls
  assessed_value           : 25221 nulls
  zip                      : 0 nulls
  year_built               : 8620 nulls
  living_area_sqft         : 12827 nulls
  lot_size_sqft            : 8138 nulls
Price source: living_area_sqft (proxy — no financial data available)
Date source: synthetic — staggered from year_built using attomid hash

Usable rows with price + date + zip: 12394
Feature columns available: ['living_area_sqft', 'lot_size_sqft', 'bedrooms', 'bathrooms', 'year_built', 'property_type', 'stories', 'garage_spaces', 'pool_flag', 'assessed_value', 'tax_amount', 'last_assessed_year', 'zip', 'county', 'state', 'latitude', 'longitude']

Repeat-sale pairs: 0
-> Using ZIP-month aggregation fallback
   Rows available for aggregation: 12394
   ZIP-

,zip,horizon_months,target,_date_key,baseline_sale_price,time_since_last_sale,living_area_sqft,lot_size_sqft,bedrooms,bathrooms,...,latitude,longitude,property_type,garage_spaces,county,state,mortgage_rate,fed_funds,unemployment,CPI
0,10001,6,-0.781496,1980-01-01,48347.0,NaN,2064.0,7406.0,NaN,NaN,...,40.748351,-73.995482,CONDOMINIUM,NaN,New York,NY,NaN,NaN,NaN,NaN
1,10001,12,0.379982,1980-01-01,48347.0,NaN,2064.0,7406.0,NaN,NaN,...,40.748351,-73.995482,CONDOMINIUM,NaN,New York,NY,NaN,NaN,NaN,NaN
2,10001,36,-0.505802,1980-01-01,48347.0,NaN,2064.0,7406.0,NaN,NaN,...,40.748351,-73.995482,CONDOMINIUM,NaN,New York,NY,NaN,NaN,NaN,NaN
3,90210,6,0.230795,1980-01-01,3026.5,NaN,3979.5,16333.0,4.0,4.0,...,34.090887,-118.406273,SFR,No Garage,Los Angeles,CA,NaN,NaN,NaN,NaN
4,90210,12,0.188171,1980-01-01,3026.5,NaN,3979.5,16333.0,4.0,4.0,...,34.090887,-118.406273,SFR,No Garage,Los Angeles,CA,NaN,NaN,NaN,NaN


In [10]:
# Cell 6 — Preprocessing with 70/15/15 time-based split

import json
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('/content/training_merged.csv')

TARGET    = 'target'
DROP_COLS = ['_date_key', 'baseline_sale_date', 'prior_sale_date', 'attomid']
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
df = df.dropna(subset=[TARGET])
df = df[df[TARGET].between(-0.9, 5.0)]

# ── Drop columns that are entirely null — imputer cannot handle them ───────────
all_null = [c for c in df.columns if df[c].isna().all()]
if all_null:
    print(f"Dropping {len(all_null)} all-null columns: {all_null}")
    df = df.drop(columns=all_null)

# ── Build CAT/NUM col lists from what actually exists ─────────────────────────
CANDIDATE_CAT = ['property_type', 'zip', 'county', 'state', 'garage_spaces']
CAT_COLS = [c for c in CANDIDATE_CAT if c in df.columns]
NUM_COLS = [c for c in df.columns if c not in set(CAT_COLS + [TARGET])]

print(f"Total rows      : {len(df)}")
print(f"Numeric features: {NUM_COLS}")
print(f"Categoric feats : {CAT_COLS}")

for c in CAT_COLS:
    df[c] = df[c].astype(str).fillna('unknown')

# ── Time-based 70 / 15 / 15 split ─────────────────────────────────────────────
n       = len(df)
tr_end  = int(n * 0.70)
val_end = int(n * 0.85)

df_tr  = df.iloc[:tr_end].copy()
df_val = df.iloc[tr_end:val_end].copy()
df_te  = df.iloc[val_end:].copy()

print(f"\nSplit sizes:")
print(f"  Train      : {len(df_tr):>6} rows  ({len(df_tr)/n*100:.1f}%)")
print(f"  Validation : {len(df_val):>6} rows  ({len(df_val)/n*100:.1f}%)")
print(f"  Test       : {len(df_te):>6} rows  ({len(df_te)/n*100:.1f}%)")

# ── Impute numerics (fit on train only) ───────────────────────────────────────
num_imputer = SimpleImputer(strategy='median')
X_tr_num  = pd.DataFrame(num_imputer.fit_transform(df_tr[NUM_COLS]),
                          columns=NUM_COLS, index=df_tr.index)
X_val_num = pd.DataFrame(num_imputer.transform(df_val[NUM_COLS]),
                          columns=NUM_COLS, index=df_val.index)
X_te_num  = pd.DataFrame(num_imputer.transform(df_te[NUM_COLS]),
                          columns=NUM_COLS, index=df_te.index)

# ── One-hot encode categoricals (fit on train only) ───────────────────────────
if CAT_COLS:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_tr_cat  = pd.DataFrame(ohe.fit_transform(df_tr[CAT_COLS]),
                              columns=ohe.get_feature_names_out(CAT_COLS),
                              index=df_tr.index)
    X_val_cat = pd.DataFrame(ohe.transform(df_val[CAT_COLS]),
                              columns=ohe.get_feature_names_out(CAT_COLS),
                              index=df_val.index)
    X_te_cat  = pd.DataFrame(ohe.transform(df_te[CAT_COLS]),
                              columns=ohe.get_feature_names_out(CAT_COLS),
                              index=df_te.index)
    X_tr  = pd.concat([X_tr_num,  X_tr_cat],  axis=1)
    X_val = pd.concat([X_val_num, X_val_cat], axis=1)
    X_te  = pd.concat([X_te_num,  X_te_cat],  axis=1)
else:
    ohe   = None
    X_tr  = X_tr_num
    X_val = X_val_num
    X_te  = X_te_num

y_tr  = df_tr[TARGET].values
y_val = df_val[TARGET].values
y_te  = df_te[TARGET].values

FEATURE_NAMES = list(X_tr.columns)
print(f"\nFeature matrix shape:")
print(f"  Train      : {X_tr.shape}")
print(f"  Validation : {X_val.shape}")
print(f"  Test       : {X_te.shape}")

# ── Save preprocessing metadata ───────────────────────────────────────────────
meta = {
    'num_cols':            NUM_COLS,
    'cat_cols':            CAT_COLS,
    'feature_names':       FEATURE_NAMES,
    'num_imputer_medians': num_imputer.statistics_.tolist(),
    'ohe_categories':      {c: list(cats) for c, cats in zip(CAT_COLS, ohe.categories_)}
                           if ohe else {}
}
with open('/content/preprocess_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print("Saved /content/preprocess_meta.json")

Dropping 4 all-null columns: ['time_since_last_sale', 'assessed_value', 'tax_amount', 'last_assessed_year']
Total rows      : 4334
Numeric features: ['horizon_months', 'baseline_sale_price', 'living_area_sqft', 'lot_size_sqft', 'bedrooms', 'bathrooms', 'year_built', 'stories', 'pool_flag', 'latitude', 'longitude', 'mortgage_rate', 'fed_funds', 'unemployment', 'CPI']
Categoric feats : ['property_type', 'zip', 'county', 'state', 'garage_spaces']

Split sizes:
  Train      :   3033 rows  (70.0%)
  Validation :    650 rows  (15.0%)
  Test       :    651 rows  (15.0%)

Feature matrix shape:
  Train      : (3033, 29)
  Validation : (650, 29)
  Test       : (651, 29)
Saved /content/preprocess_meta.json


In [12]:
# Cell 7 — Model training on GPU (XGBoost 2.0+ syntax)
import subprocess
import numpy as np
import xgboost
import joblib
from sklearn.metrics import mean_absolute_error

# Verify GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500])
print(f"XGBoost version: {xgboost.__version__}")

model = xgboost.XGBRegressor(
    tree_method="hist",       # XGBoost 2.0+: 'gpu_hist' is replaced by 'hist' + device='cuda'
    device="cuda",            # XGBoost 2.0+: use this instead of predictor='gpu_predictor'
    objective="reg:squarederror",
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=30,
    random_state=42
)

print("Training XGBoost on GPU (early stopping on validation set)...")
model.fit(
    X_tr, y_tr,
    eval_set=[
        (X_tr,  y_tr),    # track train loss
        (X_val, y_val)    # early stopping monitors this
    ],
    verbose=50
)

print(f"\nBest iteration: {model.best_iteration}")

# ── Evaluate on all three splits ──────────────────────────────────────────────
def evaluate(name, X, y):
    pred = model.predict(X)
    mae  = mean_absolute_error(y, pred)
    mape = np.mean(np.abs((y - pred) / (np.abs(y) + 1e-8))) * 100
    print(f"  {name:12s} — MAE: {mae:.4f}  |  MAPE: {mape:.2f}%")

print("\n=== Evaluation ===")
evaluate("Train",      X_tr,  y_tr)
evaluate("Validation", X_val, y_val)
evaluate("Test",       X_te,  y_te)

# ── Save model in two formats ─────────────────────────────────────────────────
model.save_model('/content/appreciation_xgb.json')
print("\nSaved /content/appreciation_xgb.json")

joblib.dump(model, '/content/appreciation_xgb.joblib')
print("Saved /content/appreciation_xgb.joblib")

Sun Mar  1 11:58:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       
XGBoost version: 3.2.0
Training XGBoost on GPU (early stopping on validation set)...
[0]	validation_0-rmse:0.80917	validation_1-rmse:0.68672
[50]	validation_0-rmse:0.66162	validation_1-rmse:0.61863
[91]	validation_0-rmse:0.62115	validation_1-rmse:0.61717

Best iteration: 61

=== Evaluation ===
  Train        — MAE: 0.3812  |  MAPE: 8040611.67%
  Validation   — MAE: 0.4354  |  MAPE: 11008907.12%
  Test         — MAE: 0.6487  |  MAPE: 215.48%

Saved /content/appreciation_xgb.json
Saved /content/a

In [13]:
from google.colab import files
files.download('/content/appreciation_xgb.joblib')
files.download('/content/appreciation_xgb.json')
files.download('/content/preprocess_meta.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 8 — Scenario Perturbation

def predict_scenarios(row: dict, horizon_months: int) -> dict:
    """
    Predict property appreciation under best/avg/worst macro scenarios.

    Parameters
    ----------
    row : dict
        Property feature dict including macro values.
    horizon_months : int
        Prediction horizon (6, 12, or 36 months).

    Returns
    -------
    dict with keys: best, avg, worst
    """
    PERTURBATIONS = {
        'avg':   {},
        'best':  {'mortgage_rate': -1.0, 'fed_funds': -0.5, 'unemployment': -0.5},
        'worst': {'mortgage_rate': +1.0, 'fed_funds': +0.5, 'unemployment': +0.5}
    }

    results = {}
    for scenario_name, perturbations in PERTURBATIONS.items():
        perturbed = row.copy()
        perturbed['horizon_months'] = horizon_months
        for macro_key, delta in perturbations.items():
            if macro_key in perturbed and perturbed[macro_key] is not None:
                perturbed[macro_key] = float(perturbed[macro_key]) + delta

        row_df = pd.DataFrame([perturbed])
        for col in numeric_cols:
            if col not in row_df.columns:
                row_df[col] = np.nan
        for col in cat_cols:
            if col not in row_df.columns:
                row_df[col] = 'missing'

        X_scenario = preprocessor.transform(row_df[feature_cols])
        pred = model.predict(X_scenario)[0]
        results[scenario_name] = float(pred)

    return results


# Demonstration
latest_macro = df_fred_clean.sort_values('macro_date').iloc[-1].to_dict()

if len(X_test_raw) > 0:
    sample_row = X_test_raw.iloc[0].to_dict()
else:
    sample_row = {c: np.nan for c in numeric_cols}
    for c in cat_cols:
        sample_row[c] = 'missing'

# Inject latest macro values
for key in ['mortgage_rate', 'fed_funds', 'unemployment', 'cpi']:
    sample_row[key] = latest_macro.get(key, None)

print('=== Scenario Predictions for Sample Property ===')
preview_keys = ['zip', 'bedrooms', 'bathrooms', 'living_area_sqft',
                'baseline_sale_price', 'mortgage_rate', 'fed_funds', 'unemployment']
for k in preview_keys:
    if k in sample_row:
        print(f'  {k}: {sample_row[k]}')
print()

for horizon in [6, 12, 36]:
    preds = predict_scenarios(sample_row, horizon)
    print(f'Horizon: {horizon} months')
    for scenario, val in preds.items():
        print(f'  {scenario:6s}: {val*100:+.2f}% appreciation')
    print()

print('Pipeline complete! Files saved to /content/')
for f in [
    '/content/attom_raw.csv',
    '/content/fred_macros.csv',
    '/content/training_merged.csv',
    '/content/appreciation_xgb.json',
    '/content/appreciation_xgb.joblib',
    '/content/preprocess_meta.json'
]:
    print(f'  {f}')